## Customer Dataset = Data Profiling & Cleaning

In [0]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [0]:
df = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/customer.csv')
df.head(5)

In [0]:
# Data Profiling -- Customer Dataset.
# 1. Structure Assesment
print("Dataframe Shape: ", df.shape)
print("Dataframe Columns: ", df.columns)
print("Dataframe Datatypes: ", df.dtypes)
# 2. Missing Values
print("Missing Values: ", df.isnull().sum())
# 3. Duplicates
print("Duplicates: ", df.duplicated().sum().sum())
# 4. Data Types
print("Data Types: ", df.dtypes)

In [0]:
customer_silver = spark.createDataFrame(df)
customer_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Olist_Retail.Silver_Datasets.customer_silver")
print("Table Created Successfully...")

## Order Dataset - Data Profiling & Cleaning

In [0]:
df_order = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/orders.csv')
print("Dataset Read Successfull.")

In [0]:
# ==========================
# Data Profiling - Orders Dataset
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_order.shape)

print("\nColumns:")
print(df_order.columns.tolist())

print("\nData Types:")
print(df_order.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_order.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_order.isnull().sum() / len(df_order)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_order.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_order.nunique())



In [0]:
# Changing The datatype of 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date' from object to date
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    df_order[col] = pd.to_datetime(df_order[col], errors='coerce')

In [0]:
df_order.info()

In [0]:
# Check duplicates order_id
df_order[df_order.duplicated(['order_id'])]

In [0]:
missing_rows = df_order[df_order.isnull().any(axis=1)]

print(missing_rows)

In [0]:
missing_approved = df_order[df_order['order_approved_at'].isnull()]

print("Total Missing Approval Dates:", len(missing_approved))
print("\nOrder Status Distribution:")
print(missing_approved['order_status'].value_counts())

# tells us order status distribution in missing 'order_approved_at'

In [0]:
# Checking 14 delivered rows+
df_order[
    (df_order['order_approved_at'].isnull()) &
    (df_order['order_status'] == 'delivered')
]

# Data Quality Exceptions

In [0]:
missing_carrier_date = df_order[df_order['order_delivered_carrier_date'].isnull()]

print("Total Missing Carrier Dates:", len(missing_carrier_date))
print("\nOrder Status Distribution:")
print(missing_carrier_date['order_status'].value_counts())

In [0]:
df_order[
    (df_order['order_delivered_carrier_date'].isnull()) &
    (df_order['order_status'] == 'delivered')
]


In [0]:
df_order.drop(index=92643, inplace=True)

In [0]:
missing_customer_date = df_order[df_order['order_delivered_customer_date'].isnull()]

print("Total Missing customer date: ", len(missing_customer_date))
print("\nOrder Status Distribution: ")
print(missing_customer_date['order_status'].value_counts())


In [0]:
df_order[(df_order['order_delivered_customer_date'].isnull()) & (df_order['order_status'] == 'delivered')]
# Data Quality Exception

In [0]:
spark.sql("DROP TABLE IF EXISTS Olist_Retail.Silver_Datasets.order_silver")

order_silver = spark.createDataFrame(df_order)
order_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.order_silver")
print("Table created Successfully")

## Order Payment - Data Profiling & Cleaning

In [0]:
df_pay = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/order_payments.csv')
print("Data Read Successfull")

In [0]:
# ==========================
# Data Profiling - Payment Dataset
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_pay.shape)

print("\nColumns:")
print(df_pay.columns.tolist())

print("\nData Types:")
print(df_pay.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_pay.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_pay.isnull().sum() / len(df_pay)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_pay.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_pay.nunique())

# 5. Statisticsl Summary
print("\nStatistical Summary")
print(df_pay.describe())



In [0]:
spark.sql("DROP TABLE IF EXISTS Olist_Retail.Silver_Datasets.payment_silver")

payment_silver = spark.createDataFrame(df_pay)
payment_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.payment_silver")
print("Table created Successfully")

## Order Item - Data Profiling & Cleaning

In [0]:
df_oitem = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/order_items.csv')
print("Data Read Successfull")

In [0]:
df_oitem.head(3)

In [0]:
# ==========================
# Data Profiling - Order Item Dataset
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_oitem.shape)

print("\nColumns:")
print(df_oitem.columns.tolist())

print("\nData Types:")
print(df_oitem.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_oitem.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_oitem.isnull().sum() / len(df_oitem)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_oitem.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_oitem.nunique())

# 5. Statisticsl Summary
print("\nStatistical Summary")
print(df_oitem.describe())



In [0]:
# Change datatime from object to datetime
df['shipping_limit_date'] = pd.to_datetime(
    df_oitem['shipping_limit_date'],
    errors='coerce'
)

In [0]:
spark.sql("DROP TABLE IF EXISTS Olist_Retail.Silver_Datasets.oitems_silver")

order_silver = spark.createDataFrame(df_oitem)
order_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.oitems_silver")
print("Table created Successfully")

## Product Dataset - Data Profiling & Cleaning

In [0]:
df_product = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/products.csv')
print("Data Read Successfull")

In [0]:
df_product.head(3)

In [0]:
# ==========================
# Data Profiling - Order Products Dataset
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_product.shape)

print("\nColumns:")
print(df_product.columns.tolist())

print("\nData Types:")
print(df_product.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_product.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_product.isnull().sum() / len(df_product)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_product.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_product.nunique())

# 5. Statisticsl Summary
print("\nStatistical Summary")
print(df_product.describe())

In [0]:
missing_df_products = df_product[df_product['product_category_name'].isnull()]
print(len(missing_df_products))
missing_df_products.head(3)


In [0]:
missing_df_products['product_category_name'] = "Unknown"
missing_df_products


In [0]:
missing_df_products

In [0]:
products_silver = spark.createDataFrame(df_product)
products_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.products_silver")
print("Table created Successfully")

## Sellers Dataset - Data Profiling & CLeaning

In [0]:
df_sellers = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/sellers.csv')
print("Data Read Successfull....")

In [0]:
# ==========================
# Data Profiling - Sellers Dataset
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_sellers.shape)

print("\nColumns:")
print(df_sellers.columns.tolist())

print("\nData Types:")
print(df_sellers.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_sellers.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_sellers.isnull().sum() / len(df_sellers)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_sellers.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_sellers.nunique())

# 5. Describe info
print("\nSeller Dataset Info:")
print(df_sellers.info())

In [0]:
sellers_silver = spark.createDataFrame(df_sellers)
sellers_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.sellers_silver")
print("Table created Successfully")

## Geolocation Dataset - Data Profiling & Cleaning

In [0]:
df_geo = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/geolocation.csv')

In [0]:
# ==========================
# Data Profiling - Geolocation Dataset
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_geo.shape)

print("\nColumns:")
print(df_geo.columns.tolist())

print("\nData Types:")
print(df_geo.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_geo.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_geo.isnull().sum() / len(df_geo)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_geo.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_geo.nunique())

# 5. Describe info
print("\nSeller Dataset Info:")
print(df_geo.info())

In [0]:
geo_silver = spark.createDataFrame(df_geo)
geo_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.geo_silver")
print("Table created Successfully")

## Product Category Name Translation - Data Profiling & Cleaning

In [0]:
df_trans = pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/product_category_name_translation.csv')
print("Data Read Successfull")

In [0]:
# ==========================
# Data Profiling - Product Category Name Translation Dataset
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_trans.shape)

print("\nColumns:")
print(df_trans.columns.tolist())

print("\nData Types:")
print(df_trans.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_trans.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_trans.isnull().sum() / len(df_trans)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_trans.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_trans.nunique())

# 5. Describe info
print("\nSeller Dataset Info:")
print(df_trans.info())

In [0]:
trans_silver = spark.createDataFrame(df_trans)
trans_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.trans_silver")
print("Table created Successfully")

## Product Review - Data Profiling & Cleaning


In [0]:
df_review =pd.read_csv('/Volumes/olist_retail/bronze_datasets/olist_bronze_dataset/order_review.csv')
print("Data Read Succesfull")

In [0]:
# ==========================
# Data Profiling - order review
# ==========================

# 1. Structure Assessment
print("Dataset Shape:")
print(df_review.shape)

print("\nColumns:")
print(df_review.columns.tolist())

print("\nData Types:")
print(df_review.dtypes)

# 2. Missing Values
print("\nMissing Values:")
print(df_review.isnull().sum())

# Missing Percentage
print("\nMissing Percentage:")
print((df_review.isnull().sum() / len(df_review)) * 100)

# 3. Duplicate Records
print("\nDuplicate Rows:")
print(df_review.duplicated().sum())

# 4. Unique Values
print("\nUnique Values Per Column:")
print(df_review.nunique())

# 5. Describe info
print("\nSeller Dataset Info:")
print(df_review.info())

In [0]:
review_silver = spark.createDataFrame(df_review)
review_silver.write \
    .format("delta") \
    .mode('overwrite') \
    .saveAsTable("Olist_Retail.Silver_Datasets.review_silver")
print("Table created Successfully")